# 05 - Evaluate LLaMEA Champions (N=10 Independent Runs)

This notebook:
1. Loads problem-specific champions from `data/champions.json` (generated by Notebook 04).
2. Executes each champion **N=10 independent times** on target BBOB problems across multiple dimensions and noise levels.
3. Uses configured noise strategies (`MultiplicativeNoiseStrategy`).
4. Attaches IOH Analyzer via `problem.attach_analyzer(...)` to output IOH `.dat` performance files to `data/ioh_logs/f{p_id}_{dim}D_std{noise_std}/llamea_champion_std{noise_std}/`.


In [1]:
import sys
import json
import numpy as np
from pathlib import Path

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from infra.problems.bbob import BBOBProblem
from domain.services.noise_strategy import MultiplicativeNoiseStrategy, NoNoiseStrategy
from synthesis.execution import AlgorithmExecutor

# ── Experiment Configuration ─────────────────────────────
CHAMPIONS_PATH  = PROJECT_ROOT / 'data' / 'champions.json'
IOH_LOGS_DIR    = PROJECT_ROOT / 'data' / 'ioh_logs'
DIMS            = [2, 3]             # Dimensions to evaluate
NOISE_STDS      = [0.0, 0.05, 0.1]  # Noise std levels to evaluate
N_RUNS          = 10                 # Independent runs per config
BUDGET          = 100000             # Function evaluation budget
TIMEOUT_SECONDS = 30.0
# ─────────────────────────────────────────────────────────

print(f'Champions JSON: {CHAMPIONS_PATH}')
print(f'IOH Logs Output: {IOH_LOGS_DIR}')
print(f'Target Dimensions: {DIMS}')
print(f'Target Noise STDs: {NOISE_STDS}')
print(f'Runs per champion: {N_RUNS}')
print(f'Budget: {BUDGET} evaluations')


ModuleNotFoundError: No module named 'synthesis.executor'

## 1. Load Champions JSON

In [ ]:
if not CHAMPIONS_PATH.exists():
    raise FileNotFoundError(f'Champions file not found at {CHAMPIONS_PATH}. Please run Notebook 04 first.')

with open(CHAMPIONS_PATH, 'r') as f:
    champions = json.load(f)

print(f'Loaded {len(champions)} champion configuration(s):')
for pid, info in champions.items():
    print(f"  f{pid}: {info['algorithm_name']} (from Exp #{info['experiment_id']})")

## 2. Execute Champion Evaluation Benchmark

In [ ]:
executor = AlgorithmExecutor(timeout_seconds=TIMEOUT_SECONDS)

for pid_str, info in champions.items():
    p_id = int(pid_str)
    code_file = PROJECT_ROOT / info['code_path'] if not Path(info['code_path']).is_absolute() else Path(info['code_path'])
    
    if not code_file.exists():
        print(f'[WARN] Code file for f{p_id} not found at {code_file}. Skipping.')
        continue
        
    code_content = code_file.read_text(encoding='utf-8')
    algo_name = info['algorithm_name']
    
    for dim in DIMS:
        for noise_std in NOISE_STDS:
            out_dir = IOH_LOGS_DIR / f'f{p_id}_{dim}D_std{noise_std}'
            out_dir.mkdir(parents=True, exist_ok=True)
            
            folder_name = f'llamea_champion_std{noise_std}'
            print()
            print(f'=== Evaluating Champion for f{p_id} ({dim}D, noise={noise_std}): {algo_name} (N={N_RUNS} runs, Budget={BUDGET}) ===')
            
            noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
            problem = BBOBProblem(
                problem_id=p_id,
                dim=dim,
                instance_id=1,
                noise_strategy=noise_strat,
            )
            
            # Attach IOH logger once for all N runs
            problem.attach_analyzer(
                log_dir=out_dir,
                folder_name=folder_name,
                algorithm_name='LLaMEA Champion',
                algorithm_info=f'dim={dim}, noise_std={noise_std}, exp={info.get("experiment_id", "N/A")}',
            )
            
            clean_errors = []
            for run_idx in range(1, N_RUNS + 1):
                problem.reset()
                try:
                    best_x, best_y = executor.execute_algorithm(
                        code=code_content,
                        name=algo_name,
                        dim=dim,
                        problem=problem.get_objective_fn(),
                        budget=BUDGET,
                    )
                    
                    if best_x is not None:
                        clean_val = problem.eval_clean(best_x)
                        clean_err = abs(clean_val - problem.true_optimum)
                        clean_errors.append(clean_err)
                        evals = problem.evaluations
                        print(f'  Run {run_idx:2d}/{N_RUNS}: evals={evals}, final clean error={clean_err:.6e}')
                    else:
                        print(f'  Run {run_idx:2d}/{N_RUNS}: returned best_x is None')
                except Exception as e:
                    print(f'  Run {run_idx:2d}/{N_RUNS} execution failed: {e}')
                    
            # Safely close logger after all N runs complete
            problem.close_logger()
            
            if clean_errors:
                med_err = float(np.median(clean_errors))
                print(f'  f{p_id} ({dim}D, noise={noise_std}) Champion Median Clean Error across {len(clean_errors)} runs: {med_err:.6e}')
        
print()
print('Champion evaluations complete!')
